# FireWatch · Notebook 01 — XGBoost Daily ERC Regression
**Runs only after a real source-aligned training table exists.** Does not use binary `future_7d_hotspot` labels. Native target = documented gridMET Energy Release Component (NFDRS, fuel model G).

Expected CSV: `~/firewatch/data/processed/training_cell_days_erc.csv` with `cell_id`, `local_date`, `target_erc` and the numerical features listed below. **Do not fabricate training labels** to satisfy this notebook. Before exporting predictions, add regional/seasonal percentile calibration, source-version manifests, spatial evaluation, weather forecast reconciliation, and SHAP checks.

In [ ]:
from pathlib import Path
from datetime import date
import csv, json, math
import numpy as np
ROOT = Path.home()/"firewatch"
TABLE = ROOT/"data/processed/training_cell_days_erc.csv"
MODEL_DIR = ROOT/"models"
FEATURES = ["temp_max_c","rh_min_pct","wind_max_ms","precip_mm","precip_7d_mm","elevation_m","day_of_year_sin","day_of_year_cos"]
print("Native XGBoost + true ERC target; input path:",TABLE)

## 1. Fail closed unless we have real, verified daily ERC + matching weather
Missing table or fields is a blocker, not an invitation to train on the old `future_7d_hotspot` labels. Independent NASA VIIRS hotspots remain external plausibility validation only.

In [ ]:
if not TABLE.is_file():
    raise FileNotFoundError(f"Missing REAL ERC+weather table: {TABLE}. Complete Notebook 00 and data ingestion first.")
with TABLE.open(newline="") as f:
    reader=csv.DictReader(f)
    required={"cell_id","local_date","target_erc","target_source","weather_source","grid_version"}|set(FEATURES)
    missing=required-set(reader.fieldnames or [])
    if missing: raise ValueError(f"Source-aligned table missing required columns: {sorted(missing)}")
    rows=list(reader)
print("Rows:",len(rows))

## 2. Validate and split chronologically
Recommended initial split: train on 2023–24, tune on 2025, reserve available 2026 dates as final holdout. No random cell-day split; spatial evaluation is a separate required gate.

In [ ]:
import collections
seen=set(); data={"train":[],"validation":[],"test":[]}
for row in rows:
    key=(row["cell_id"],row["local_date"])
    if key in seen: raise ValueError(f"Duplicate grid/date: {key}")
    seen.add(key)
    d=date.fromisoformat(row["local_date"])
    # Guardrail: do not silently accept undocumentable label provenance.
    if not row["target_source"].strip().lower().startswith("gridmet"):
        raise ValueError(f"Unexpected label provenance on {key}: {row['target_source']}")
    if not row["grid_version"] or not row["weather_source"]:
        raise ValueError(f"Missing lineage for {key}")
    vals=[float(row[k]) for k in FEATURES]
    y=float(row["target_erc"])
    if not np.all(np.isfinite(vals)) or not math.isfinite(y) or y<0:
        continue
    s="train" if d.year in (2023,2024) else "validation" if d.year==2025 else "test" if d.year==2026 else None
    if s: data[s].append((vals,y))
for k,v in data.items(): print(k,len(v))
if not data["train"] or not data["validation"]:
    raise ValueError("Need complete real 2023–24 train and 2025 validation periods")
X_train=np.asarray([x for x,y in data["train"]],dtype=np.float32)
y_train=np.asarray([y for x,y in data["train"]],dtype=np.float32)
X_val=np.asarray([x for x,y in data["validation"]],dtype=np.float32)
y_val=np.asarray([y for x,y in data["validation"]],dtype=np.float32)

## 3. Native XGBoost training; no sklearn dependency
Runs on IBM LinuxONE only after `xgboost` imports natively, and after input-validation succeeds. One regressor predicts daily ERC; seven daily forecast feature rows produce seven different daily ERC estimates.

In [ ]:
import xgboost as xgb
print("XGBoost version:",xgb.__version__)
train_matrix=xgb.DMatrix(X_train,label=y_train,feature_names=FEATURES)
val_matrix=xgb.DMatrix(X_val,label=y_val,feature_names=FEATURES)
params={"objective":"reg:squarederror","eval_metric":"rmse","tree_method":"hist","max_depth":5,"eta":0.05,"subsample":0.8,"colsample_bytree":0.8,"lambda":5,"seed":42,"nthread":4}
booster=xgb.train(params,train_matrix,num_boost_round=600,evals=[(train_matrix,"train"),(val_matrix,"validation")],early_stopping_rounds=40,verbose_eval=50)
print("Best iteration:",booster.best_iteration)

## 4. Evaluate validation predictions against a constant-mean baseline
These metrics reflect accuracy in native ERC index units. They are **not** ignition probabilities. Do not publish final holdout metrics until a non-overlapping holdout exists.

In [ ]:
best=int(booster.best_iteration)+1
pred=booster.predict(val_matrix,iteration_range=(0,best))
constant=np.full(len(y_val),float(y_train.mean()))
def score(y,p):
    return {"mae":float(np.mean(np.abs(y-p))),"rmse":float(np.sqrt(np.mean((y-p)**2)))}
metrics={"target":"gridMET_ERC_fuel_model_G","features":FEATURES,"validation_year":2025,"xgboost":score(y_val,pred),"mean_baseline":score(y_val,constant),"n_val":int(len(y_val)),"best_trees":best,"final_2026_test":"NOT_EVALUATED"}
print(json.dumps(metrics,indent=2))

## 5. Export versioned model and metadata — experimental, not production
The backend's prior seven-day binary hotspot contract is NOT compatible with daily ERC predictions. Do not replace `risk.geojson` without the teams approving a versioned contract.

In [ ]:
MODEL_DIR.mkdir(exist_ok=True,parents=True)
booster.save_model(str(MODEL_DIR/"erc_xgb_experimental.json"))
(MODEL_DIR/"erc_xgb_experimental_metrics.json").write_text(json.dumps(metrics,indent=2)+"\n")
print("Saved experimental model and evaluation report under",MODEL_DIR)

## Remaining release gates
- Check time-zone conventions and precipitation lag windows.
- Add a geographic block holdout and untouched 2026 temporal test if available.
- Compare archived Open-Meteo forecast predictors versus historical gridMET features.
- Test native LinuxONE loading/inference and XGBoost contribution sums (`pred_contribs=True`).
- Derive calibrated *relative ERC percentiles* from a documented region/season reference population; never label as wildfire ignition probability.
- Publish only after backend/frontend approve daily ERC contract and grid version.
